# FahMai RAG Starter Enhanced

Base จาก `Starter Kit FahMai RAG.py` และเพิ่ม embedding ที่แรงขึ้น, smarter chunking, chunk-size tuning, openthaigpt, prompt engineering และ reranking

## 1) Install Dependencies (optional)


In [ ]:
# %pip install -U sentence-transformers pythainlp rank-bm25 requests python-dotenv


## 2) Imports


In [ ]:
import os
import csv
import re
import json
import time
from dataclasses import dataclass
from pathlib import Path
from collections import defaultdict

import numpy as np
import requests

from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from pythainlp.tokenize import word_tokenize

try:
    from pythainlp.tokenize import sent_tokenize
except Exception:
    sent_tokenize = None

try:
    from sentence_transformers import CrossEncoder
    HAS_CROSS_ENCODER = True
except Exception:
    CrossEncoder = None
    HAS_CROSS_ENCODER = False


## 3) Config + Prompt


In [ ]:
# =========================
# CONFIG
# =========================
N_QUESTIONS = 100
DATA_DIR = "."  # e.g. "/content/data" on Colab
KB_DIR = f"{DATA_DIR}/knowledge_base"

LLM_MODEL = "openthaigpt"  # required by task
EMBED_MODEL_NAME = "intfloat/multilingual-e5-large"

USE_RERANK = True
RERANK_MODEL_NAME = "BAAI/bge-reranker-v2-m3"
RERANK_TOP_N = 32
RERANK_WEIGHT = 0.35

# Chunk size tuning (run all indexes and fuse)
CHUNK_SETTINGS = [
    {"name": "s256", "size": 256, "overlap": 64},
    {"name": "s512", "size": 512, "overlap": 128},
    {"name": "s1024", "size": 1024, "overlap": 256},
]

FETCH_K_PER_INDEX = 24
TOP_K_CONTEXT = 8
RRF_K = 60

API_SLEEP_SEC = 0.25
THAILLM_TIMEOUT = 120
MAX_RETRIES = 6

USE_E5_PREFIX = True


# Try to read API key from Colab userdata first, then env var
THAILLM_API_KEY = ""
try:
    from google.colab import userdata
    THAILLM_API_KEY = userdata.get("ThaiLLM")
except Exception:
    THAILLM_API_KEY = os.environ.get("THAILLM_API_KEY", "")

if not THAILLM_API_KEY:
    THAILLM_API_KEY = os.environ.get("THAILLM_API_KEY", "")


SYSTEM_PROMPT = """คุณคือผู้ช่วยตอบคำถามหลายตัวเลือกของร้านฟ้าใหม่
กติกา:
1) ใช้เฉพาะข้อมูลจากบริบทที่ให้มา
2) วิเคราะห์ตัวเลือก 1-8 แบบย่อ แล้วเลือกคำตอบที่ตรงที่สุด
3) ถ้าข้อมูลไม่พอสำหรับคำถามที่ยังเกี่ยวกับร้าน -> เลือก 9
4) ถ้าคำถามไม่เกี่ยวกับร้านฟ้าใหม่ -> เลือก 10
5) ส่งออกเป็น JSON บรรทัดเดียวเท่านั้น:
{"answer": <1-10>, "confidence": <0-1>, "evidence": ["C1","C2"], "reason": "..."}
"""

FEW_SHOT = """
ตัวอย่างสั้น:
- ถ้า context ระบุชัดเจนตรงกับหนึ่งในตัวเลือก -> เลือกข้อนั้น
- ถ้าถามฟีเจอร์สินค้าของร้าน แต่ context ไม่มีข้อมูลยืนยัน -> ตอบ 9
- ถ้าถามเรื่องนอกโดเมนร้าน (เช่น ท่องเที่ยวทั่วไป) -> ตอบ 10
"""


## 4) API + Parsing + Data Loading


In [ ]:
@dataclass
class Chunk:
    chunk_id: int
    source: str
    category: str
    section: str
    setting_name: str
    text: str


class RetrievalIndex:
    def __init__(self, name, chunk_ids, chunks, embed_model):
        self.name = name
        self.chunk_ids = chunk_ids
        self.chunks = chunks
        self.embed_model = embed_model

        texts = [self.chunks[i].text for i in self.chunk_ids]
        enc_texts = [prefix_for_e5(t, is_query=False) for t in texts]
        self.embeddings = self.embed_model.encode(
            enc_texts,
            batch_size=64,
            show_progress_bar=True,
            normalize_embeddings=True,
        )

        tokenized = [tokenize_thai(t) for t in texts]
        self.bm25 = BM25Okapi(tokenized)

    def dense_retrieve(self, query, k):
        q = prefix_for_e5(query, is_query=True)
        q_emb = self.embed_model.encode([q], normalize_embeddings=True)
        scores = np.dot(self.embeddings, q_emb.T).flatten()
        idx = np.argsort(scores)[::-1][:k]
        return [(self.chunk_ids[i], float(scores[i])) for i in idx]

    def bm25_retrieve(self, query, k):
        tokens = tokenize_thai(query)
        scores = np.array(self.bm25.get_scores(tokens))
        idx = np.argsort(scores)[::-1][:k]
        return [(self.chunk_ids[i], float(scores[i])) for i in idx]


def prefix_for_e5(text, is_query):
    if USE_E5_PREFIX and "e5" in EMBED_MODEL_NAME.lower():
        return f"{'query' if is_query else 'passage'}: {text}"
    return text


def ask_llm(messages, model=LLM_MODEL, max_retries=MAX_RETRIES):
    url = f"http://thaillm.or.th/api/{model}/v1/chat/completions"
    headers = {"Content-Type": "application/json", "apikey": THAILLM_API_KEY}
    payload = {
        "model": "/model",
        "messages": messages,
        "max_tokens": 1024,
        "temperature": 0,
    }

    for attempt in range(max_retries):
        try:
            resp = requests.post(url, headers=headers, json=payload, timeout=THAILLM_TIMEOUT)
            if resp.status_code == 429:
                wait = min(2 ** attempt, 30)
                print(f"Rate limited, wait {wait}s")
                time.sleep(wait)
                continue
            if 500 <= resp.status_code < 600:
                wait = min(2 ** attempt, 20)
                print(f"Server error {resp.status_code}, wait {wait}s")
                time.sleep(wait)
                continue

            resp.raise_for_status()
            return resp.json()["choices"][0]["message"]["content"].strip()
        except requests.exceptions.RequestException as e:
            wait = min(2 ** attempt, 20)
            print(f"Error: {e}, retry in {wait}s")
            time.sleep(wait)

    return None


def parse_json_answer(text):
    if text is None:
        return None, 0.0, [], ""

    clean = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    clean = re.sub(r"^```json\s*", "", clean, flags=re.IGNORECASE)
    clean = re.sub(r"^```\s*", "", clean)
    clean = re.sub(r"\s*```$", "", clean)

    for cand in re.findall(r"\{[\s\S]*?\}", clean):
        try:
            obj = json.loads(cand)
            ans = int(obj.get("answer"))
            if 1 <= ans <= 10:
                conf = float(obj.get("confidence", 0.5))
                conf = max(0.0, min(1.0, conf))
                evidence = obj.get("evidence", []) if isinstance(obj.get("evidence", []), list) else []
                reason = str(obj.get("reason", "")).strip()
                return ans, conf, evidence, reason
        except Exception:
            continue

    m = re.search(r"\b([1-9]|10)\b", clean)
    if m:
        return int(m.group(1)), 0.3, [], "fallback parse"

    return None, 0.0, [], "parse failed"


def load_questions(path):
    qs = []
    with open(path, encoding="utf-8") as f:
        for row in csv.DictReader(f):
            choices = {str(i): row[f"choice_{i}"] for i in range(1, 11)}
            qs.append({"id": int(row["id"]), "question": row["question"], "choices": choices})
    return qs


def load_documents(kb_dir):
    docs = []
    for fp in sorted(Path(kb_dir).rglob("*.md")):
        text = fp.read_text(encoding="utf-8")
        rel = str(fp.relative_to(kb_dir))
        docs.append({
            "path": rel,
            "category": rel.split("/", 1)[0] if "/" in rel else "unknown",
            "text": text,
        })
    return docs


## 5) Smarter Chunking + Multi-Chunk-Size Index


In [ ]:
def split_sections(md_text):
    lines = md_text.splitlines()
    title = ""
    section = "ข้อมูลทั่วไป"
    buf = []
    out = []

    def flush():
        nonlocal buf
        body = "\n".join(buf).strip()
        if body:
            out.append((section, body))
        buf = []

    for ln in lines:
        if ln.startswith("# "):
            title = ln[2:].strip()
            continue
        if re.match(r"^#{2,6}\s+", ln):
            flush()
            section = re.sub(r"^#{2,6}\s+", "", ln).strip()
            continue
        if ln.strip() == "---":
            continue
        buf.append(ln)
    flush()

    if not out and md_text.strip():
        out = [("ข้อมูลทั่วไป", md_text.strip())]
    return title, out


def split_sentences(text):
    if sent_tokenize is not None:
        try:
            sents = [s.strip() for s in sent_tokenize(text) if s.strip()]
            if sents:
                return sents
        except Exception:
            pass
    parts = re.split(r"(?<=[.!?？！。])\s+|\n", text)
    return [p.strip() for p in parts if p.strip()]


def semantic_pack(text, size, overlap):
    paragraphs = [p.strip() for p in re.split(r"\n\s*\n", text) if p.strip()]
    if not paragraphs:
        return []

    units = []
    soft_limit = int(size * 0.75)
    for p in paragraphs:
        if len(p) <= soft_limit:
            units.append(p)
        else:
            for s in split_sentences(p):
                if len(s) <= size:
                    units.append(s)
                else:
                    units.extend(simple_char_chunks(s, soft_limit, 0))

    chunks = []
    cur = ""
    for u in units:
        if not cur:
            cur = u
            continue
        if len(cur) + 1 + len(u) <= size:
            cur = cur + "\n" + u
        else:
            chunks.append(cur)
            if overlap > 0:
                tail = cur[-overlap:]
                cur = (tail + "\n" + u).strip()
                if len(cur) > size:
                    cur = u
            else:
                cur = u
    if cur.strip():
        chunks.append(cur.strip())
    return chunks


def simple_char_chunks(text, size, overlap):
    if len(text) <= size:
        return [text]
    out = []
    step = max(1, size - overlap)
    i = 0
    while i < len(text):
        out.append(text[i:i + size])
        i += step
    return out


def build_chunks_with_setting(documents, setting):
    size = setting["size"]
    overlap = setting["overlap"]
    name = setting["name"]
    out = []

    for doc in documents:
        title, sections = split_sections(doc["text"])
        for sec_name, sec_text in sections:
            packed = semantic_pack(sec_text, size=size, overlap=overlap)
            for block in packed:
                text = (
                    f"[SOURCE] {doc['path']}\n"
                    f"[TITLE] {title or Path(doc['path']).stem}\n"
                    f"[SECTION] {sec_name}\n\n"
                    f"{block.strip()}"
                )
                if len(text.strip()) < 80:
                    continue
                out.append({
                    "source": doc["path"],
                    "category": doc["category"],
                    "section": sec_name,
                    "setting": name,
                    "text": text.strip(),
                })
    return out


def build_all_chunks(documents):
    chunks = []
    cid = 0
    for setting in CHUNK_SETTINGS:
        part = build_chunks_with_setting(documents, setting)
        for c in part:
            chunks.append(Chunk(
                chunk_id=cid,
                source=c["source"],
                category=c["category"],
                section=c["section"],
                setting_name=c["setting"],
                text=c["text"],
            ))
            cid += 1
    return chunks


def tokenize_thai(text):
    toks = word_tokenize(text, engine="newmm")
    out = []
    for t in toks:
        t = t.strip().lower()
        if not t:
            continue
        if re.fullmatch(r"[^\wก-๙]+", t):
            continue
        out.append(t)
    return out


def normalize_scores(x):
    x = np.asarray(x, dtype=np.float32)
    if len(x) == 0:
        return x
    lo, hi = float(x.min()), float(x.max())
    if abs(hi - lo) < 1e-9:
        return np.full_like(x, 0.5)
    return (x - lo) / (hi - lo)


def build_indices(chunks, embed_model):
    grouped = defaultdict(list)
    for ch in chunks:
        grouped[ch.setting_name].append(ch.chunk_id)

    indices = []
    for setting in CHUNK_SETTINGS:
        name = setting["name"]
        idx = RetrievalIndex(name, grouped[name], chunks, embed_model)
        indices.append(idx)
    return indices


def maybe_build_reranker():
    if not USE_RERANK:
        return None
    if not HAS_CROSS_ENCODER:
        print("[WARN] CrossEncoder not available, skip rerank")
        return None
    try:
        return CrossEncoder(RERANK_MODEL_NAME)
    except Exception as e:
        print(f"[WARN] Failed to load reranker: {e}")
        return None



## 6) Hybrid Retrieval + Reranking + Prompt Build


In [ ]:
def hybrid_retrieve(query, choices, indices, chunks, reranker=None, top_k=TOP_K_CONTEXT):
    # combine query with choice hints (lightweight expansion)
    choice_hint = " ".join(choices[str(i)] for i in range(1, 9))
    q_full = f"{query}\n{choice_hint}"

    rrf = defaultdict(float)

    for idx_obj in indices:
        dense_hits = idx_obj.dense_retrieve(q_full, k=FETCH_K_PER_INDEX)
        bm25_hits = idx_obj.bm25_retrieve(q_full, k=FETCH_K_PER_INDEX)

        for rank, (cid, _) in enumerate(dense_hits, 1):
            rrf[cid] += 1.20 / (RRF_K + rank)
        for rank, (cid, _) in enumerate(bm25_hits, 1):
            rrf[cid] += 1.00 / (RRF_K + rank)

    ranked = sorted(rrf.items(), key=lambda x: x[1], reverse=True)

    # Reranking on top candidates
    if reranker is not None and ranked:
        cand = ranked[:RERANK_TOP_N]
        pairs = [(query, chunks[cid].text) for cid, _ in cand]
        ce_scores = np.array(reranker.predict(pairs, batch_size=16, show_progress_bar=False), dtype=np.float32)
        ce_scores = normalize_scores(ce_scores)

        for (cid, _), sc in zip(cand, ce_scores):
            rrf[cid] += RERANK_WEIGHT * float(sc)
        ranked = sorted(rrf.items(), key=lambda x: x[1], reverse=True)

    # de-duplicate nearly same chunks from different chunk settings
    seen = set()
    final_ids = []
    for cid, _ in ranked:
        key = (chunks[cid].source, chunks[cid].text[:200])
        if key in seen:
            continue
        seen.add(key)
        final_ids.append(cid)
        if len(final_ids) >= top_k:
            break

    retrieved = [chunks[cid] for cid in final_ids]
    return retrieved


def build_rag_prompt(question, choices, retrieved_chunks):
    context = "\n\n".join(
        f"[C{i+1}] source={c.source} | chunk={c.setting_name} | section={c.section}\n{c.text}"
        for i, c in enumerate(retrieved_chunks)
    )
    choices_text = "\n".join(f"{k}. {v}" for k, v in choices.items())

    return (
        f"บริบท:\n{context}\n\n"
        f"คำถาม: {question}\n\n"
        f"ตัวเลือก:\n{choices_text}\n\n"
        f"{FEW_SHOT}\n"
        "ตอบ JSON ตาม schema เท่านั้น"
    )


def run_pipeline(questions, indices, chunks, reranker=None, n=N_QUESTIONS):
    predictions = {}

    for i, q in enumerate(questions[:n], 1):
        retrieved = hybrid_retrieve(
            query=q["question"],
            choices=q["choices"],
            indices=indices,
            chunks=chunks,
            reranker=reranker,
            top_k=TOP_K_CONTEXT,
        )

        prompt = build_rag_prompt(q["question"], q["choices"], retrieved)
        raw = ask_llm([
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ], model=LLM_MODEL)

        ans, conf, evidence, reason = parse_json_answer(raw)
        if ans is None or ans < 1 or ans > 10:
            ans = 9

        predictions[q["id"]] = ans
        print(f"Q{q['id']:>3} -> pred={ans} conf={conf:.2f}")

        time.sleep(API_SLEEP_SEC)

    return predictions


## 7) Main Pipeline


In [ ]:
def main():
    if not THAILLM_API_KEY:
        raise RuntimeError("Missing THAILLM_API_KEY (env or Colab userdata 'ThaiLLM')")

    questions = load_questions(f"{DATA_DIR}/questions.csv")
    documents = load_documents(KB_DIR)

    print(f"Loaded {len(questions)} questions")
    print(f"Loaded {len(documents)} documents")

    print("Building chunks with multiple chunk-size settings...")
    chunks = build_all_chunks(documents)
    print(f"Total chunks: {len(chunks)}")

    print(f"Loading embed model: {EMBED_MODEL_NAME}")
    embed_model = SentenceTransformer(EMBED_MODEL_NAME)

    print("Building retrieval indices...")
    indices = build_indices(chunks, embed_model)

    print("Loading reranker...")
    reranker = maybe_build_reranker()

    print(f"Running pipeline with LLM model: {LLM_MODEL}")
    preds = run_pipeline(questions, indices, chunks, reranker=reranker, n=N_QUESTIONS)

    out_path = "submission_enhanced.csv"
    with open(out_path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["id", "answer"])
        for q in questions:
            w.writerow([q["id"], preds.get(q["id"], 9)])

    print(f"Wrote {out_path}")


## 8) Run


In [ ]:
main()
